In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

In [25]:
# load the datasets
train = pd.read_csv("fashion-mnist_train.csv")
test = pd.read_csv("fashion-mnist_test.csv")

print(train.shape)
print(test.shape)

(60000, 785)
(10000, 785)


In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [27]:
X_train = train.drop('label' , axis=1)
y_train = train['label']
X_test = test.drop('label' , axis=1)
y_test = test['label']

In [28]:
# convert to tensors
X_train = torch.from_numpy(np.array(X_train)).to(dtype=torch.float32)
X_test = torch.from_numpy(np.array(X_test)).to(dtype=torch.float32)
y_train = torch.from_numpy(np.array(y_train)).to(dtype=torch.long)
y_test = torch.from_numpy(np.array(y_test)).to(dtype=torch.long)

In [29]:
# standardize the data from 0 to 1

X_train = X_train/255.
X_test = X_test/255.

In [47]:
# create dataset and dataloader
from torch.utils.data import Dataset , DataLoader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return self.features[idx] , self.labels[idx]

In [48]:
train_dataset = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test , y_test)

In [53]:
train_loader = DataLoader(train_dataset , batch_size=32 , shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size=32 , shuffle=True,pin_memory=True)

In [54]:
## Model Building
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self , num_features):

        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features , 128),
            nn.ReLU(),
            nn.Linear(128 , 64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    # forward pass

    def forward(self,features):
        out = self.network(features)
        return out

In [55]:
# Parameters
learning_rate = 0.1
epochs=100

# create the model Object
model = MyModel(X_train.shape[1])
model = model.to(device) # this saves my model in GPU

# create loss

criterion = nn.CrossEntropyLoss()

# optimizer

optimizer = optim.SGD(model.parameters() , lr = learning_rate)

In [56]:
for batch_features, batch_labels in train_loader:
    print(batch_features.shape)
    print(batch_labels.shape)
    break

torch.Size([32, 784])
torch.Size([32])


In [57]:
# Let us now build the back propagation code
import time
start_time = time.time()
for epoch in range(epochs):
    total_loss = 0
    for batch_features,batch_labels in train_loader:
        # save them in GPU
        batch_features,batch_labels = batch_features.to(device ), batch_labels.to(device)

        # forward pass

        outputs = model(batch_features)

        # calc loss

        loss = criterion(outputs , batch_labels) # this automatically converts logits to probs using softmax

        # remove the gradients
        optimizer.zero_grad()

        # back propagate
        loss.backward()

        # update the weights
        optimizer.step()

        # find the sum of loss in each epoch
        total_loss += loss
    print(f'Loss at epoch {epoch+1} ->{total_loss/len(train_loader)}')
print(time.time()-start_time)

Loss at epoch 1 ->0.6113702654838562
Loss at epoch 2 ->0.41537413001060486
Loss at epoch 3 ->0.37146416306495667
Loss at epoch 4 ->0.34640610218048096
Loss at epoch 5 ->0.32343804836273193
Loss at epoch 6 ->0.3082643151283264
Loss at epoch 7 ->0.29384925961494446
Loss at epoch 8 ->0.28307220339775085
Loss at epoch 9 ->0.2750661075115204
Loss at epoch 10 ->0.2655438780784607
Loss at epoch 11 ->0.25787052512168884
Loss at epoch 12 ->0.25096407532691956
Loss at epoch 13 ->0.2441025674343109
Loss at epoch 14 ->0.23688898980617523
Loss at epoch 15 ->0.23402149975299835
Loss at epoch 16 ->0.22615914046764374
Loss at epoch 17 ->0.22159263491630554
Loss at epoch 18 ->0.2174779325723648
Loss at epoch 19 ->0.21456800401210785
Loss at epoch 20 ->0.20863601565361023
Loss at epoch 21 ->0.2037954479455948
Loss at epoch 22 ->0.19979228079319
Loss at epoch 23 ->0.1948312371969223
Loss at epoch 24 ->0.19114936888217926
Loss at epoch 25 ->0.18787913024425507
Loss at epoch 26 ->0.1867280900478363
Loss at

In [58]:
# Evaluation 
# set to eval mode
model.eval()
total = 0
correct = 0
for batch_features , batch_labels in test_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')



Accuracy Score -> 0.8917
